# Task 2: Hand Tracking Setup

Sign-language speed game — Task 2 (hand pose estimation + sign classification).

Run cells top to bottom. Make sure the kernel is set to **Python (ipcv-task2)**.

**Note:** Uses MediaPipe's newer Tasks API (`HandLandmarker`), not the older `mp.solutions.hands` API. MediaPipe removed `mp.solutions` starting from version 1.0; this project pins `mediapipe==0.10.35` in `requirements.txt` because 1.0.1 currently crashes on Apple Silicon Macs during model load ([known bug](https://github.com/google-ai-edge/mediapipe/issues/6356)).

## 1. Check imports

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.core.base_options import BaseOptions

print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("NumPy:", np.__version__)

## 2. Test webcam access

Grabs a single frame to confirm the camera works and check its resolution. On macOS, the first run may trigger a camera-permission prompt for your terminal/VS Code — allow it.

In [ ]:
cap = cv2.VideoCapture(0)
ret, frame = cap.read()
cap.release()

if ret:
    print("Webcam OK. Frame shape:", frame.shape)
else:
    print("Could not read from webcam. Check camera permissions / index (try 1 instead of 0).")

## 3. Load the hand landmarker model

Uses the `.task` model file already downloaded to `../models/hand_landmarker.task`.

- `num_hands=4` supports 2 players x 2 hands each.
- `running_mode=IMAGE` is used here for simplicity (one-shot detection per frame). We can switch to `VIDEO` mode later if we want MediaPipe's built-in temporal tracking between frames.

In [ ]:
MODEL_PATH = "../models/hand_landmarker.task"

hand_options = vision.HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=vision.RunningMode.IMAGE,
    num_hands=4,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
landmarker = vision.HandLandmarker.create_from_options(hand_options)
print("HandLandmarker loaded OK")

## 4. Live hand tracking (both hands)

Opens a native OpenCV window showing your webcam feed with hand landmarks drawn on top.

- Press **q** in the video window to stop cleanly.
- If it ever hangs, interrupt the kernel (stop button) — the `finally` block below still releases the camera.

In [ ]:
HAND_CONNECTIONS = vision.HandLandmarksConnections.HAND_CONNECTIONS

# MediaPipe's handedness output is calibrated for a non-mirrored camera feed.
# Since we flip the frame for natural (mirror-style) display, MediaPipe's
# "Left"/"Right" ends up inverted relative to the player's real hand — swap it here.
HANDEDNESS_SWAP = {"Left": "Right", "Right": "Left"}

def draw_landmarks(frame, hand_landmarks_list, handedness_list):
    h, w, _ = frame.shape
    for hand_landmarks, handedness in zip(hand_landmarks_list, handedness_list):
        points = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]

        for connection in HAND_CONNECTIONS:
            start, end = connection.start, connection.end
            cv2.line(frame, points[start], points[end], (0, 200, 0), 2)
        for x, y in points:
            cv2.circle(frame, (x, y), 3, (0, 255, 0), -1)

        raw_label = handedness[0].category_name
        label = HANDEDNESS_SWAP.get(raw_label, raw_label)
        score = handedness[0].score
        wrist_x, wrist_y = points[0]
        cv2.putText(
            frame, f"{label} {score:.2f}", (wrist_x, wrist_y - 20),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2,
        )
    return frame


cap = cv2.VideoCapture(0)

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Frame grab failed, stopping.")
            break

        frame = cv2.flip(frame, 1)  # mirror for natural interaction
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

        result = landmarker.detect(mp_image)

        if result.hand_landmarks:
            frame = draw_landmarks(frame, result.hand_landmarks, result.handedness)

        cv2.imshow("Task 2 - Hand Tracking (press q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    cap.release()
    # macOS sometimes leaves the OpenCV window stuck/frozen after destroyAllWindows()
    # unless a few more frames are processed right after — this flushes the close event.
    cv2.waitKey(1)
    cv2.destroyAllWindows()
    cv2.waitKey(1)

## 5. Extract normalized landmarks (building block for classification)

This is the function you'll reuse for data collection and later for live classification: it takes MediaPipe's 21 landmarks and normalizes them (wrist-relative, scale-invariant) so the classifier doesn't care about hand distance/position in frame.

In [ ]:
def normalize_landmarks(hand_landmarks):
    """Convert a HandLandmarker hand's 21 landmarks to a flat, translation- and
    scale-invariant 63-length feature vector (21 landmarks x 3 coords).

    `hand_landmarks` is one entry from `result.hand_landmarks` (a list of
    NormalizedLandmark objects with .x, .y, .z).
    """
    coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

    # Translate: wrist (landmark 0) becomes the origin
    coords -= coords[0]

    # Scale: normalize by the distance from wrist to middle-finger MCP (landmark 9)
    scale = np.linalg.norm(coords[9])
    if scale > 1e-6:
        coords /= scale

    return coords.flatten()  # shape (63,)


print("Feature vector length should be 63.")

## Cleanup

Run this when you're done experimenting to release the model.

In [ ]:
landmarker.close()
print("Landmarker closed.")

## Next steps

1. Confirm cells 1–4 work on your machine (camera + live tracking window with landmarks).
2. Build a data-collection cell: press a key to label and save `normalize_landmarks(...)` output for each sign to `../data/`.
3. Move to `02_data_collection.ipynb` and `03_train_classifier.ipynb` (create these next).
4. Once stable, move the landmarker setup and `normalize_landmarks` into `src/hand_tracker.py` so it's importable for the team.